# <font color="#003660">Applied Machine Learning for Text Analysis (M.184.5331)</font>


# <font color="#003660">Session 1: Introduction to Natural Language Processing</font>

# <font color="#003660">Notebook 1b: Document Regression</font>

<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/dag.png"/><br></center>

<p>
<center>
<div>
    <font color="#085986"><b>By the end of this lesson, you will be able to...</b><br><br>
        ... transform raw text into a term-document matrix, and <br>
        ... train a regressor on the term-document matrix.
    </font>
</div>
</center>
</p>

# Import packages

As always, we first need to load a number of required Python packages:
- `pandas` provides high-performance, easy-to-use data structures and data analysis tools.
- `spacy` offers industrial-strength natural language processing.
- `sklearn` is the de-facto standard machine learning package in Python.

In [ ]:
%pip install -q spacy-lookups-data

In [ ]:
import pandas as pd
import spacy
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LinearRegression
from sklearn import metrics

# Load documents

Load wine reviews (Source: https://www.kaggle.com/datasets/zynicide/wine-reviews) from a csv file.

In [ ]:
corpus = pd.read_csv("https://raw.githubusercontent.com/olivermueller/amlta-2026/main/Session_01/winemag-data-130k-v2.csv")
corpus.rename(columns = {'Unnamed: 0':'index'}, inplace = True)

In [ ]:
corpus.head()

In [ ]:
corpus.shape

# Preprocess documents

Split data into training, validation, and test set.

In [ ]:
training = corpus.iloc[0:80000,].sample(n=10000, random_state=42) # sample to speed up training
validation = corpus.iloc[80000:100000,]
test = corpus.iloc[100000:,]

In [ ]:
print(training.shape)
print(validation.shape)
print(test.shape)

Perform the same NLP preprocessing steps as in Notebook 1a: tokenize the reviews with spaCy, keep only alphabetic tokens that are not stop words, and reduce them to their lemmas (using a fast lookup lemmatizer).

In [ ]:
nlp = spacy.blank("en")
nlp.add_pipe("lemmatizer", config={"mode": "lookup"})
nlp.initialize()

def spacy_prep_df(df, text_col="description", batch_size=1000):
    texts = (str(x).lower() if pd.notna(x) and x else "" for x in df[text_col].values)  # the lookup lemmatizer is case-sensitive
    results = []
    for doc in nlp.pipe(texts, batch_size=batch_size):
        toks = [t.lemma_ for t in doc if t.is_alpha and not t.is_stop]
        results.append(" ".join(toks))
    out = df.copy()
    out[text_col + "_prep"] = results
    return out


In [ ]:
training = spacy_prep_df(training)

Display the first couple of lines of the preprocessed descriptions.

In [ ]:
training["description_prep"].head()

# Vectorize documents

Vectorization is the process of turning a collection of text documents into numerical feature vectors.

We will use the **Bag of Words (BoW)** model for vectorization. In the BoW model, a corpus of documents is represented by a matrix with one row per document and one column per word occurring in the corpus. The cell values will either be simple frequency counts (How often does a word appear in a document?), or the term frequency (tf) times the inverse document frequency (idf) of a term. The idea of tf-idf is to scale down the impact of words that occur very frequently in a given corpus and that are therefore less informative than features that occur only in a small fraction of the corpus. Note that the BoW model completely ignores information about the position and sequences of the words in the document.

In `sklearn`, the [`CountVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html#sklearn.feature_extraction.text.CountVectorizer) creates a term-document matrix with (normalized) term frequencies and the [`TfIdfVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html#sklearn.feature_extraction.text.TfidfVectorizer) creates a term-document matrix with tf-idf weighting.

In [ ]:
count_vect = CountVectorizer(min_df=10)

Apply the CountVectorizer object to the review texts of the training set.

In [ ]:
X_training = count_vect.fit_transform(training["description_prep"])

Display an extract of the generated term-document matrix

In [ ]:
X_training.shape

In [ ]:
X_training[0:20,0:20].todense()

Store the labels that we want to predict in a separate variable.

In [ ]:
y_training = training["points"]
y_training.describe()

# Train regressor on training set

Fit a linear regression model with the term-document matrix as the features and the wine quality (i.e., `points` variable) as the numerical label.

In [ ]:
reg = LinearRegression().fit(X_training, y_training)

Test whether the regression model is working by predicting the quality of a short fake review. We apply the same NLP preprocessing steps and reuse the `count_vect` object to generate features in the same way as we did for the training set.

In [ ]:
doc_new = {'description': ['This is a spectacular, magnificent, and majestic wine. Awesome!']}
doc_new_df = pd.DataFrame.from_dict(doc_new)

In [ ]:
doc_new_df_prep = spacy_prep_df(doc_new_df)
doc_new_df_prep

In [ ]:
X_new = count_vect.transform(doc_new_df_prep["description_prep"])
predicted = reg.predict(X_new)
predicted

# Evaluate prediction error on validation set

Before trying to predict the labels for the official test set, we evaluate the prediction error of our model on the validation set. Again, we apply the same NLP preprocessing steps, reuse the `count_vect` object, and store `X` and `y` in separate data structures.

In [ ]:
validation = spacy_prep_df(validation)

In [ ]:
X_validation = count_vect.transform(validation["description_prep"])
y_validation = validation["points"]

Call the predict function of our model with the validation data and calculate the mean absolute error (MAE), i.e., by how many points our predictions are off on average.

In [ ]:
predictions_validation = reg.predict(X_validation)
print(metrics.mean_absolute_error(y_validation, predictions_validation))

To judge whether this is good or bad, compare it with a naive benchmark that always predicts the mean points of the training set.

In [ ]:
print(metrics.mean_absolute_error(y_validation, [y_training.mean()] * len(y_validation)))